In [1]:
import numpy as np

In [21]:
import pandas as pd

df = pd.DataFrame({
    "building_name": ["Marina Gate Tower 1", "Ocean Heights"],
    "master_project": ["Marina Gate", "Dubai Marina"],
    "area_name": ["Dubai Marina", "Dubai Marina"]
})

In [20]:
df = pd.read_csv('Transactions.csv')
df.head()

,transaction_id,procedure_id,trans_group_id,trans_group_ar,trans_group_en,procedure_name_ar,procedure_name_en,instance_date,property_type_id,property_type_ar,...,rooms_en,has_parking,procedure_area,actual_worth,meter_sale_price,rent_value,meter_rent_price,no_of_parties_role_1,no_of_parties_role_2,no_of_parties_role_3
0,3-9-2006-163,9,3,هبات,Gifts,هبه,Grant,16-10-2006,4,فيلا,...,NaN,0,3162.42,12000000.0,3794.56,NaN,NaN,3.0,1.0,0.0
1,3-9-2019-2944,9,3,هبات,Gifts,هبه,Grant,13-11-2019,1,أرض,...,NaN,0,209.09,916659.0,4384.04,NaN,NaN,2.0,4.0,0.0
2,2-13-1999-347,13,2,رهون,Mortgages,تسجيل رهن,Mortgage Registration,22-03-1999,1,أرض,...,NaN,0,1062.72,1200000.0,1129.18,NaN,NaN,1.0,1.0,0.0
3,2-13-2001-547,13,2,رهون,Mortgages,تسجيل رهن,Mortgage Registration,23-07-2001,2,مبنى,...,NaN,0,1393.55,3500000.0,2511.57,NaN,NaN,5.0,1.0,0.0
4,2-13-2020-9477,13,2,رهون,Mortgages,تسجيل رهن,Mortgage Registration,30-11-2020,2,مبنى,...,NaN,0,278.71,2500000.0,8969.90,NaN,NaN,1.0,1.0,0.0


In [30]:
def build_query(row):
    parts = [
        row.get("building_name", ""),
        row.get("master_project", ""),
        row.get("area_name", ""),
        "Dubai",
        "UAE"
    ]

    parts = [p for p in parts if pd.notna(p) and p != ""]
    return ", ".join(parts)

df["query"] = df.apply(build_query, axis=1)

In [31]:
df['query']

0    Marina Gate Tower 1, Marina Gate, Dubai Marina...
1    Ocean Heights, Dubai Marina, Dubai Marina, Dub...
Name: query, dtype: str

In [38]:
import requests
import time

def geocode(query):
    
    url = "https://nominatim.openstreetmap.org/search"
    
    params = {
        "q": query,
        "format": "json",
        "limit": 1
    }
    query2 = ', '.join(query.split(',')[1:])
    print(query2)
    params = {
        "q": query2,
        "format": "json",
        "limit": 1
    }
    
    headers = {
        "User-Agent": "house-price-model"
    }

    try:
        r = requests.get(url, params=params, headers=headers)
        data = r.json()
        
        if len(data) > 0:
            return float(data[0]["lat"]), float(data[0]["lon"])
        else:
            r = requests.get(url, params=params, headers=headers)
            data = r.json()
            if len(data) > 0:
                return float(data[0]["lat"]), float(data[0]["lon"])
    
    except Exception as e:
        print(e)
        pass
    
    return None, None

In [39]:
cache = {}

In [40]:
from tqdm import tqdm

lats = []
lngs = []

for q in tqdm(df["query"]):
    
    if q in cache:
        lat, lng = cache[q]
    else:
        lat, lng = geocode(q)
        cache[q] = (lat, lng)
        time.sleep(1)   # required for Nominatim rate limit

    lats.append(lat)
    lngs.append(lng)

df["lat"] = lats
df["lng"] = lngs

  0%|                                                                                | 0/2 [00:00<?, ?it/s]

 Marina Gate,  Dubai Marina,  Dubai,  UAE


 50%|████████████████████████████████████                                    | 1/2 [00:01<00:01,  1.66s/it]

 Dubai Marina,  Dubai Marina,  Dubai,  UAE


100%|████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.50s/it]


In [41]:
df.head()

,building_name,master_project,area_name,query,query2,lat,lng
0,Marina Gate Tower 1,Marina Gate,Dubai Marina,"Marina Gate Tower 1, Marina Gate, Dubai Marina...","Marina Gate, Dubai Marina, Dubai, UAE",25.086972,55.147091
1,Ocean Heights,Dubai Marina,Dubai Marina,"Ocean Heights, Dubai Marina, Dubai Marina, Dub...","Dubai Marina, Dubai Marina, Dubai, UAE",25.082135,55.144592


In [42]:
df.iloc[0]['query']

'Marina Gate Tower 1, Marina Gate, Dubai Marina, Dubai, UAE'